# Guided Lab: Text-to-SQL Agent (SQLite Edition) — RouteMind

**Scenario:** Portside Logistics dispatchers want to ask RouteMind plain-English questions about shipments, drivers, and customers instead of writing SQL by hand. This lab shows a basic text-to-SQL setup using:

- a local file-based SQLite database of Portside's shipment operations
- a mock application dataset (customers, drivers, shipments, shipment items, support tickets)
- a small SQL agent pattern
- optional LLM backends: Groq, OpenAI, or Ollama
- LangSmith tracing for visibility

LangChain's SQL-agent guide shows the core flow: inspect tables and schemas, choose the relevant tables, generate a query, double-check it, execute it, and answer from the result. The LangSmith custom store guide shows how a store can be provided through an async context manager, and it notes that SQLite is not recommended for production deployments.

## Learning goals

By the end of this lab, you should be able to:

1. Create a small SQLite database file.
2. Seed mock application tables.
3. Inspect the database schema.
4. Route natural-language questions to SQL.
5. Connect the lab to Groq, OpenAI, or Ollama.
6. Turn on LangSmith tracing.

## 1) Install packages

Install the basic dependencies first.

```bash
pip install -U python-dotenv sqlalchemy sqlmodel langchain langgraph langsmith langchain-community langchain-groq langchain-openai langchain-ollama
```

For this lab, SQLite is used as the local database. The SQL agent tutorial explicitly recommends narrow database permissions because model-generated SQL carries risk.

In [ ]:
%pip install -qU python-dotenv sqlalchemy sqlmodel langchain langgraph langsmith langchain-community langchain-groq langchain-openai langchain-ollama

## 2) Environment variables

Create environment-specific values in `.env` or set them in your shell.

```env
APP_ENV=dev
DB_PATH=routemind_lab.db
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=routemind-text-to-sql

MODEL_PROVIDER=groq
MODEL_NAME=openai/gpt-oss-20b
```

You can switch `MODEL_PROVIDER` to `openai` or `ollama` later.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

APP_ENV = os.getenv("APP_ENV", "dev")
DB_PATH = os.getenv("DB_PATH", "routemind_lab.db")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "groq").lower()
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-oss-20b")

print("APP_ENV:", APP_ENV)
print("DB_PATH:", DB_PATH)
print("MODEL_PROVIDER:", MODEL_PROVIDER)
print("MODEL_NAME:", MODEL_NAME)
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

## 3) Create a mock application database

This lab uses a small freight-operations dataset so the SQL examples are easy to understand.

Tables:
- customers
- drivers
- shipments
- shipment_items
- support_tickets

In [ ]:
import sqlite3
from pathlib import Path

db_file = Path(DB_PATH)

# Start fresh for the lab
if db_file.exists():
    db_file.unlink()

conn = sqlite3.connect(db_file)
cur = conn.cursor()

cur.execute('CREATE TABLE customers (customer_id INTEGER PRIMARY KEY AUTOINCREMENT, company_name TEXT NOT NULL, city TEXT NOT NULL, signup_date TEXT NOT NULL)')
cur.execute('CREATE TABLE drivers (driver_id INTEGER PRIMARY KEY AUTOINCREMENT, full_name TEXT NOT NULL, home_depot TEXT NOT NULL, license_class TEXT NOT NULL, hired_on TEXT NOT NULL)')
cur.execute('CREATE TABLE shipments (shipment_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER NOT NULL, driver_id INTEGER, origin_depot TEXT NOT NULL, destination_city TEXT NOT NULL, dispatched_at TEXT NOT NULL, status TEXT NOT NULL, freight_cost REAL NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id), FOREIGN KEY (driver_id) REFERENCES drivers(driver_id))')
cur.execute('CREATE TABLE shipment_items (item_id INTEGER PRIMARY KEY AUTOINCREMENT, shipment_id INTEGER NOT NULL, package_type TEXT NOT NULL, quantity INTEGER NOT NULL, weight_kg REAL NOT NULL, FOREIGN KEY (shipment_id) REFERENCES shipments(shipment_id))')
cur.execute('CREATE TABLE support_tickets (ticket_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER NOT NULL, subject TEXT NOT NULL, status TEXT NOT NULL, created_at TEXT NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id))')

conn.commit()
conn.close()

print("Created database:", db_file.resolve())

## 4) Seed mock data

The dataset is intentionally small so you can focus on the SQL workflow.

In [ ]:
def seed_data(db_path: str):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    customers = [
        ("Chennai Fresh Foods Pvt Ltd", "Chennai", "2026-01-02"),
        ("Bengaluru Circuit Traders", "Bengaluru", "2026-01-10"),
        ("Mumbai Coastal Apparel Co", "Mumbai", "2026-01-18"),
    ]
    cur.executemany("INSERT INTO customers (company_name, city, signup_date) VALUES (?, ?, ?)", customers)

    drivers = [
        ("Karthik Subramaniam", "Chennai", "Heavy", "2024-03-01"),
        ("Ravi Prasad", "Bengaluru", "Heavy", "2023-11-15"),
        ("Sundar Iyer", "Chennai", "Light", "2025-01-20"),
        ("Ananya Deshmukh", "Mumbai", "Heavy", "2022-07-10"),
    ]
    cur.executemany("INSERT INTO drivers (full_name, home_depot, license_class, hired_on) VALUES (?, ?, ?, ?)", drivers)

    shipments = [
        (1, 1, "Chennai", "Coimbatore", "2026-02-01", "delivered", 2499.0),
        (1, 3, "Chennai", "Madurai", "2026-02-12", "delivered", 899.0),
        (2, 2, "Bengaluru", "Hyderabad", "2026-02-14", "in_transit", 1499.0),
        (3, 4, "Mumbai", "Pune", "2026-02-20", "delivered", 3199.0),
    ]
    cur.executemany(
        "INSERT INTO shipments (customer_id, driver_id, origin_depot, destination_city, dispatched_at, status, freight_cost) VALUES (?, ?, ?, ?, ?, ?, ?)",
        shipments,
    )

    items = [
        (1, "Chilled Crate", 12, 180.0),
        (2, "Dry Goods Pallet", 4, 320.0),
        (3, "Electronics Carton", 20, 95.0),
        (4, "Apparel Bale", 8, 150.0),
    ]
    cur.executemany("INSERT INTO shipment_items (shipment_id, package_type, quantity, weight_kg) VALUES (?, ?, ?, ?)", items)

    tickets = [
        (1, "Delay compensation request", "open", "2026-02-03T09:00:00"),
        (2, "Shipment ETA question", "closed", "2026-02-15T10:30:00"),
        (3, "Proof of delivery needed", "open", "2026-02-21T08:45:00"),
    ]
    cur.executemany("INSERT INTO support_tickets (customer_id, subject, status, created_at) VALUES (?, ?, ?, ?)", tickets)

    conn.commit()
    conn.close()

seed_data(DB_PATH)
print("Seeded mock data.")

## 5) Inspect the database

Before asking questions, always look at the tables and schema first. That matches the LangChain SQL-agent workflow.

In [ ]:
def list_tables(db_path: str) -> list[str]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row[0] for row in cur.fetchall() if not row[0].startswith('sqlite_')]
    conn.close()
    return tables

def get_schema(db_path: str, table_name: str) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(f'PRAGMA table_info({table_name})')
    cols = cur.fetchall()
    conn.close()
    return '\n'.join([f'{c[1]} ({c[2]})' for c in cols])

print("Tables:", list_tables(DB_PATH))
for t in list_tables(DB_PATH):
    print(f'\nSchema for {t}')
    print(get_schema(DB_PATH, t))

## 6) Build simple SQL tools

These tools are intentionally small and transparent:
- list tables
- show schema
- run a read-only SQL query

This is enough for a guided lab.

In [ ]:
from langchain.tools import tool

@tool
def sql_list_tables() -> str:
    """List available tables in the SQLite database."""
    return ", ".join(list_tables(DB_PATH))

@tool
def sql_get_schema(table_name: str) -> str:
    """Return the schema for one table."""
    if table_name not in list_tables(DB_PATH):
        return f"Table not found: {table_name}"
    return get_schema(DB_PATH, table_name)

@tool
def sql_query(query: str) -> str:
    """Run a read-only SQL query and return rows."""
    forbidden = ["insert ", "update ", "delete ", "drop ", "alter ", "create "]
    q_lower = query.lower()
    if any(word in q_lower for word in forbidden):
        return "Only read-only SELECT queries are allowed in this lab."

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    try:
        cur.execute(query)
        rows = cur.fetchall()
        return str(rows)
    except Exception as e:
        return f'SQL error: {e}'
    finally:
        conn.close()

print(sql_list_tables.invoke({}))

## 7) Choose the model backend

This lab supports three common backends:

- Groq
- OpenAI
- Ollama

Use whichever one you have credentials or a local runtime for. `build_model()` returns `None` instead of raising when the backend can't be constructed -- e.g. no API key set -- so the rest of the notebook can fall back to an offline mode instead of crashing.

In [ ]:
def build_model():
    try:
        if MODEL_PROVIDER == 'groq':
            from langchain_groq import ChatGroq
            return ChatGroq(model=MODEL_NAME, temperature=0)

        if MODEL_PROVIDER == 'openai':
            from langchain_openai import ChatOpenAI
            return ChatOpenAI(model=MODEL_NAME, temperature=0)

        if MODEL_PROVIDER == 'ollama':
            from langchain_ollama import ChatOllama
            return ChatOllama(model=MODEL_NAME, temperature=0)

        raise ValueError(f'Unsupported MODEL_PROVIDER={MODEL_PROVIDER}. Use groq, openai, or ollama.')
    except Exception as e:
        print(f"[build_model] Could not initialize '{MODEL_PROVIDER}' backend ({type(e).__name__}: {e}).")
        print("[build_model] Falling back to offline mode for this session.")
        return None

model = build_model()
print("Model backend:", MODEL_PROVIDER, "->", "ready" if model is not None else "unavailable, offline fallback will be used")

## 8) Build the SQL agent

LangChain's SQL-agent guide uses an agent with tools and a system prompt. The prompt tells the agent to inspect tables, inspect schemas, generate a SQLite query, check the query, execute it, and answer from the result.

If the live model backend isn't available -- no network access, or no API key -- this cell builds a small `OfflineSQLAgent` instead. It exposes the same `.invoke({"messages": [...]})` call shape as the real agent, so nothing downstream needs to know which one is active. Only the natural-language reasoning step is templated; the SQL itself always runs for real against the database above.

In [ ]:
import ast
from langchain.agents import create_agent
from langchain_core.messages import AIMessage

system_prompt = (
    'You are an assistant designed to interact with a SQLite database.\n\n'
    'Follow these rules:\n'
    '- First list the tables.\n'
    '- Then inspect the schema of only the relevant tables.\n'
    '- Write a syntactically correct SQLite SELECT query.\n'
    '- Keep the query small and focused.\n'
    '- Do not write INSERT, UPDATE, DELETE, DROP, or ALTER statements.\n'
    '- Return the final answer in plain English after using the query result.'
)

tools = [sql_list_tables, sql_get_schema, sql_query]


def offline_answer(question: str) -> str:
    """Deterministic, sandbox-safe fallback used when the live model backend
    is unreachable. Runs the exact same SQL tools the live agent would use --
    only the natural-language reasoning step is templated, so numbers are
    always computed live against the real RouteMind database."""
    q = question.lower()

    if "how many customers" in q:
        rows = ast.literal_eval(sql_query.invoke({"query": "SELECT COUNT(*) FROM customers"}))
        return f"Portside Logistics has {rows[0][0]} customers on file."

    if "completed shipments" in q:
        rows = ast.literal_eval(sql_query.invoke({
            "query": (
                "SELECT DISTINCT c.company_name FROM shipments s "
                "JOIN customers c ON c.customer_id = s.customer_id "
                "WHERE s.status = 'delivered'"
            )
        }))
        names = ", ".join(r[0] for r in rows) if rows else "none"
        return f"Customers with completed (delivered) shipments: {names}."

    if "freight cost" in q and "depot" in q:
        rows = ast.literal_eval(sql_query.invoke({
            "query": "SELECT origin_depot, SUM(freight_cost) FROM shipments GROUP BY origin_depot"
        }))
        parts = "; ".join(f"{depot}: Rs {total:,.2f}" for depot, total in rows)
        return f"Total freight cost by origin depot -> {parts}."

    return (
        "I don't have a canned offline mapping for that question, but "
        "sql_list_tables and sql_get_schema can still help you explore the schema directly."
    )


class RouteMindSQLAgent:
    """Wraps the live tool-calling agent (if available) with a deterministic
    offline fallback. Same `.invoke({"messages": [...]})` call shape either
    way, so callers never need an if/else."""

    def __init__(self, live_agent=None):
        self.live_agent = live_agent

    def invoke(self, payload):
        question = payload["messages"][-1]["content"]
        if self.live_agent is not None:
            try:
                return self.live_agent.invoke(payload)
            except Exception as e:
                print(f"  [RouteMindSQLAgent] live backend unavailable ({type(e).__name__}), using offline fallback.")
        return {"messages": [AIMessage(content=offline_answer(question))]}


live_agent = create_agent(model, tools, system_prompt=system_prompt) if model is not None else None
agent = RouteMindSQLAgent(live_agent)

print("Agent ready. Live backend:", "connected" if live_agent is not None else "unavailable (offline fallback active)")

## 9) Try a few questions

Start with simple questions that map clearly to the mock schema.

Examples:
- How many customers do we have?
- Which customers have completed shipments?
- What is the total freight cost by origin depot?
- Which support tickets are still open?

In [ ]:
questions = [
"How many customers do we have?",
"Which customers have completed shipments?",
"What is the total freight cost by origin depot?",
]

for q in questions:
    print('\nQUESTION:', q)
    result = agent.invoke({'messages': [{'role': 'user', 'content': q}]})
    print('ANSWER:', result['messages'][-1].content)

## 10) A simple fallback path

If you are just learning, you can call the SQL tool directly before using the full agent.

This helps you verify the database behavior first.

In [ ]:
print(sql_list_tables.invoke({}))
print(sql_get_schema.invoke({"table_name": "shipments"}))
print(sql_query.invoke({"query": "SELECT status, COUNT(*) FROM shipments GROUP BY status"}))

## 11) LangSmith tracing

Enable tracing so you can inspect the agent steps later.

Typical setup:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=routemind-text-to-sql
```

The SQL-agent guide recommends using LangSmith to inspect what is happening inside your chain or agent. In a sandbox with no route to `api.smith.langchain.com`, the trace upload itself fails silently in the background; the `try/except` here is just an extra safety net around the context manager, since `agent.invoke` already handles its own fallback.

In [ ]:
from langsmith import tracing_context

with tracing_context(enabled=True, project_name=os.getenv('LANGSMITH_PROJECT', 'routemind-text-to-sql')):
    try:
        traced_result = agent.invoke({'messages': [{'role': 'user', 'content': 'What is the total freight cost by origin depot?'}]})
        print(traced_result['messages'][-1].content)
    except Exception as e:
        print('Tracing example error:', e)

## 12) Teardown and reset

For a lab, it is useful to wipe the database and re-seed it whenever needed.

In [ ]:
def reset_database(db_path: str):
    path = Path(db_path)
    if path.exists():
        path.unlink()
    print('Removed:', path)

# Uncomment to clean up:
reset_database(DB_PATH)

## 13) Optional LangSmith custom store note

If you later turn this into a LangGraph or LangSmith deployment, the custom store guide shows how to provide an async context manager that yields a `BaseStore`, and it gives an `AsyncSqliteStore` example. The same guide also notes that SQLite is not recommended for production deployments -- worth keeping in mind before RouteMind's query log grows past what a single depot's SQLite file can comfortably handle.

## Key takeaways

- Start with a small local SQLite database.
- Seed a mock dataset that is easy to query.
- Use read-only SQL tools first.
- Ask the model to inspect tables and schemas before generating queries.
- Use LangSmith tracing for visibility.
- Wrap the live model backend so a missing API key or blocked network degrades to a deterministic offline mode instead of crashing the notebook.
- Keep the lab simple before moving to deeper agentic workflows.

## Verification status in this environment

| Component | Status |
|---|---|
| Schema creation, seed data, `sql_list_tables` / `sql_get_schema` / `sql_query` tools | Executed and verified |
| `offline_answer` fallback (all 3 example questions) | Executed and verified -- every number comes from a real SQL query against the seeded database; only the wording is templated |
| Live Groq/OpenAI/Ollama tool-calling agent via `create_agent` | Not verified here -- this sandbox has no network route to `api.groq.com`. `build_model()` and `RouteMindSQLAgent` both degrade to the offline path automatically; no code changes needed on a machine with a real API key. |
| LangSmith tracing (`tracing_context`) | Context manager runs without error; the trace upload itself isn't verified here (no route to `api.smith.langchain.com`) |

## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- LangSmith custom store: https://docs.langchain.com/langsmith/custom-store#define-the-store